# JEVDB: try semantic SQL

Find **five clearly positive movie reviews** using an ordinary SQL query with a natural-language condition.

Run the cells from top to bottom in a **CPU runtime**. You need your own `JEV_API_KEY` from [TypeSafe](https://docs.typesafe.ai/models).
The first run builds JEVDB from source and can take a few minutes.

The demo starts with **100 reviews**. You can change the input size, condition, and decision threshold below.
Executing the query sends review text to the Jev API and uses your account's quota. Each rerun may make new API calls.

## 1. Install JEVDB

Install the build tools and DuckDB **1.4.3**, then build the extension from the public JEVDB repository.
Run this section once per runtime. A new Colab runtime needs a fresh installation.
The setup code is collapsed by default; expand a cell to inspect it.

In [ ]:
%%bash
# @title Install build tools
set -euo pipefail
apt-get update -qq
apt-get install -y -qq build-essential cmake ninja-build libssl-dev git curl

In [ ]:
# @title Install Python packages
%pip install -q "duckdb==1.4.3" pandas

In [ ]:
# @title Build JEVDB
from pathlib import Path
import subprocess

work_dir = Path.cwd() / "jevdb_colab"
work_dir.mkdir(exist_ok=True)
repo_dir = work_dir / "JEVDB"

if not (repo_dir / ".git").exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/PDAIS/JEVDB.git", str(repo_dir),
    ], check=True)

print("JEVDB commit:", subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True,
).strip())

deps_dir = repo_dir / "build" / "dependencies"
deps_dir.mkdir(parents=True, exist_ok=True)
if not (deps_dir / "duckdb-1.4.3" / "CMakeLists.txt").exists():
    archive = deps_dir / "duckdb-v1.4.3.tar.gz"
    subprocess.run([
        "curl", "--fail", "--location", "--retry", "2",
        "https://codeload.github.com/duckdb/duckdb/tar.gz/refs/tags/v1.4.3",
        "--output", str(archive),
    ], check=True)
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(deps_dir)], check=True)

subprocess.run(["bash", "scripts/build.sh"], cwd=repo_dir, check=True)
extension_path = repo_dir / "build/release/extension/jevdb/jevdb.duckdb_extension"
print("Build complete:", extension_path.name)

In [ ]:
# @title Load JEVDB
import ctypes
import duckdb
import _duckdb
from IPython.display import Code, display

if duckdb.__version__ != "1.4.3":
    raise RuntimeError("Restart the runtime and rerun the notebook to load DuckDB 1.4.3.")


def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"


if "con" in globals():
    con.close()
# Make the Python engine symbols visible to the C++ extension on Linux.
duckdb_library = ctypes.CDLL(_duckdb.__file__, mode=ctypes.RTLD_GLOBAL)
con = duckdb.connect(config={"allow_unsigned_extensions": True})
con.execute(f"LOAD {sql_literal(extension_path)}")
print(f"JEVDB is ready in DuckDB {duckdb.__version__}.")

## 2. Configure your API key

In Colab, open **Secrets** (the key icon in the left sidebar), add a secret named `JEV_API_KEY`, and enable notebook access.
If no secret is available, the cell asks for the key using a hidden input. The key is read from the environment by JEVDB.

In [ ]:
import os
from getpass import getpass

api_key = os.environ.get("JEV_API_KEY", "")
if not api_key:
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        try:
            api_key = userdata.get("JEV_API_KEY")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
if not api_key:
    api_key = getpass("JEV_API_KEY: ").strip()
if not api_key:
    raise ValueError("Enter a JEV_API_KEY before continuing.")
os.environ["JEV_API_KEY"] = api_key
del api_key

con.execute("CREATE OR REPLACE SECRET jev (TYPE jev, PROVIDER env)")
con.execute("SET jevdb_backend = 'jev'")
con.execute("SET jevdb_model = 'jev-1.13.0'")
con.execute("SET jevdb_threads = 20")
con.execute("SET jevdb_max_retries = 2")
con.execute("SET threads = 1")
print("Jev is configured. No model request has been sent yet.")

# Jev 1.13 list price: https://docs.typesafe.ai/models (checked 2026-10-08).
input_price_usd_per_million = 0.042


## 3. Load movie reviews

Load 2,000 movie reviews (about 655 KB). The query uses `reviewId` and `reviewText`;
sentiment labels are kept separately for evaluating the default condition.

Data source: [SemBench](https://github.com/SemBench/SemBench/blob/c814e3807e72d4cf876b852b17e77f3cc94575c2/files/movie/data/sf_2000/Reviews.csv).

In [ ]:
from urllib.request import urlretrieve
import pandas as pd

sembench_revision = "c814e3807e72d4cf876b852b17e77f3cc94575c2"
data_url = (
    f"https://raw.githubusercontent.com/SemBench/SemBench/{sembench_revision}"
    "/files/movie/data/sf_2000/Reviews.csv"
)
data_path = work_dir / "Reviews.csv"
urlretrieve(data_url, data_path)
reference_labels = pd.read_csv(data_path, usecols=["reviewId", "scoreSentiment"], dtype=str)
con.execute("""
    CREATE OR REPLACE TABLE all_reviews AS
    SELECT reviewId, reviewText
    FROM read_csv(?, all_varchar = true)
""", [str(data_path)])
print("Reviews loaded:", con.execute("SELECT count(*) FROM all_reviews").fetchone()[0])
display(con.execute("SELECT * FROM all_reviews LIMIT 5").df())

## 4. Run a semantic query

Find the IDs of five clearly positive movie reviews. Change the input size, condition, or threshold,
then run the cell below to see the SQL, matching reviews, and execution summary together.

The default uses 100 reviews; choose **2,000** for the full input.
`LIMIT 5` caps the returned rows; the input size controls how many reviews are eligible for model evaluation.
The default condition also reports F1 against the reference labels, using the query's limit of five.
A custom condition displays **N/A** for F1 because it needs its own reference labels.

In [ ]:
# @title Run the query
num_reviews = 100 # @param {type:"slider", min:50, max:2000, step:50}
condition = "The following movie review is clearly positive." # @param {type:"string"}
threshold = 0.5 # @param {type:"slider", min:0.0, max:1.0, step:0.05}

from time import perf_counter
from IPython.display import HTML

if not 1 <= num_reviews <= 2000:
    raise ValueError("Choose between 1 and 2,000 reviews.")
if not 0 <= threshold <= 1:
    raise ValueError("The threshold must be between 0 and 1.")
if not condition.strip():
    raise ValueError("Enter a condition to evaluate.")

con.execute("CREATE OR REPLACE TABLE reviews AS SELECT * FROM all_reviews LIMIT ?", [int(num_reviews)])
con.execute(f"SET jevdb_threshold = {float(threshold)}")
query = f"""
SELECT reviewId
FROM reviews
WHERE jev_holds({sql_literal(condition)}, reviewText)
LIMIT 5;
""".strip()
display(Code(query, language="sql"))

started = perf_counter()
results = con.execute(query).df()
elapsed = perf_counter() - started
stats = con.execute("""
    SELECT requests, records, cache_hits, input_tokens, retries, errors, send_ms
    FROM jevdb_stats()
""").df()
estimated_cost_usd = int(stats.iloc[0]["input_tokens"]) * input_price_usd_per_million / 1_000_000

# Reference labels are used after the semantic query, never as model input.
precision = recall = f1 = None
if condition.strip() == "The following movie review is clearly positive.":
    labels = reference_labels.head(int(num_reviews))
    gold_ids = set(labels.loc[labels["scoreSentiment"].eq("POSITIVE"), "reviewId"].dropna())
    predicted_ids = set(results["reviewId"].head(5).dropna())
    hits = len(predicted_ids & gold_ids)
    precision = hits / len(predicted_ids) if predicted_ids else float(not gold_ids)
    recall = min(hits, 5) / min(5, len(gold_ids)) if gold_ids else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

summary = {
    "Returned rows": str(len(results)),
    "Query time": f"{elapsed:.2f} s",
    "API requests": str(int(stats.iloc[0]["requests"])),
    "Estimated API cost": f"${estimated_cost_usd:.8f}",
    "F1 (limit 5)": f"{f1:.3f}" if f1 is not None else "N/A",
}
display(HTML(
    '<div style="display:flex;flex-wrap:wrap;gap:24px;margin:16px 0">' +
    ''.join(f'<div><div>{label}</div><strong style="font-size:1.3em">{value}</strong></div>'
            for label, value in summary.items()) + '</div>'
))

# Add review text for display without changing the query's returned rows.
review_lookup = con.execute("SELECT DISTINCT reviewId, reviewText FROM reviews").df()
matching_reviews = results.merge(review_lookup, on="reviewId", how="left", validate="many_to_one")
if results.empty:
    print("No reviews matched. Try a lower threshold or a different condition.")
with pd.option_context("display.max_colwidth", None):
    display(matching_reviews)

scoring_note = (
    f"Precision: {precision:.3f}; recall (limit 5): {recall:.3f}. "
    "Scores compare distinct returned IDs against positive reference IDs in the selected input. "
    "Recall uses min(5, number of positive reference IDs) as its denominator."
    if f1 is not None else
    "F1 is unavailable for this custom condition; the supplied labels describe positive sentiment."
)
display(HTML(
    '<details><summary>Query statistics and scoring</summary>' +
    stats.to_html(index=False) + '<p>' + scoring_note + '</p>' +
    '<p>API requests count batches; records are counted after deduplication. '
    f'Estimated cost uses reported input tokens at ${input_price_usd_per_million:g} per million; '
    'Jev output tokens are free. This is a list-price estimate, excluding unreported usage and account-specific discounts. '
    '<a href="https://docs.typesafe.ai/models" target="_blank" rel="noopener">Pricing</a> · '
    '<a href="https://github.com/SemBench/SemBench/blob/c814e3807e72d4cf876b852b17e77f3cc94575c2/src/scenario/movie/evaluation/evaluate.py#L108" '
    'target="_blank" rel="noopener">Scoring definition</a></p></details>'
))

## Try a different condition

For example, change the condition to `The following movie review criticizes the acting.` and rerun the query cell.
Choose 2,000 reviews to query the full dataset.

More examples: [SQL functions](https://github.com/PDAIS/JEVDB/blob/main/docs/functions.md) · [Settings](https://github.com/PDAIS/JEVDB/blob/main/docs/settings.md).